# Data Scraping

This notebook contains code to scrap data on weather, UK bank holidays and covid times etc.

Different datasets need to be collected, cleaned and joined:

Variable | Unit | Description | Source
---------|------|----------|----------
Footfall| | Average number of visitors in area for a given day. | [HUQ](https://huq.io/insights/footfall-data/)
Cos_weekday_num| | | 
Sin_weekday_num| | | 
Cos_month_num| | | 
Sin_month_num| | | 
Cos_week_of_year| | | 
Sin_week_of_year| | | 
Year| | | 
bank_hol| |Whether or not that day is a bank holiday (0= No, 1= Yes)|[Kaggle](https://www.kaggle.com/datasets/shivd24coder/uk-national-holidays-dataset)
Covid times| |Whether or not that day was during covid restrictions (0= No, 1= Yes)|
Precipitation | mm | Sum of daily precipitation (including rain, showers and snowfall) | [Open Meteo API](https://open-meteo.com/en/docs/historical-weather-api?bounding_box=-90,-180,90,180&hourly=&daily=temperature_2m_mean,precipitation_sum,wind_speed_10m_max)
Temperature | °C | Average daily air temperature at 2 meters above ground | [Open Meteo API](https://open-meteo.com/en/docs/historical-weather-api?bounding_box=-90,-180,90,180&hourly=&daily=temperature_2m_mean,precipitation_sum,wind_speed_10m_max)
Wind Speed | km/h | Maximum wind speed on a day | [Open Meteo API](https://open-meteo.com/en/docs/historical-weather-api?bounding_box=-90,-180,90,180&hourly=&daily=temperature_2m_mean,precipitation_sum,wind_speed_10m_max)

In [134]:
pip install matplotlib geopandas numpy

Note: you may need to restart the kernel to use updated packages.


In [135]:
#Import packages
import pandas as pd
import seaborn as sns
import matplotlib
import geopandas as gpd
import numpy as np

In [136]:
#Load footfall data
footfall = pd.read_csv(r"C:\Users\qxnq723\Desktop\Project 1\Coding\Bradford Analysis\footfall_Met")
footfall.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2470 entries, 0 to 2469
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Unnamed: 0                         2470 non-null   int64  
 1   datestamp                          2470 non-null   object 
 2   estimated_actual_footfall          2306 non-null   float64
 3   estimated_actual_footfall_rolling  2470 non-null   int64  
 4   year                               2470 non-null   int64  
 5   month                              2470 non-null   int64  
 6   weekday                            2470 non-null   int64  
 7   week_of_year                       2470 non-null   int64  
 8   Sin_weekday                        2470 non-null   float64
 9   Cos_weekday                        2470 non-null   float64
 10  Sin_week_of_year                   2470 non-null   float64
 11  Cos_week_of_year                   2470 non-null   float

In [137]:
#Convert datestamp to datetime
footfall['datestamp'] = pd.to_datetime(footfall['datestamp'])

## Bank Holidays
Bank holiday data was collected from Kaggle.

In [138]:
bank_hols = pd.read_csv(r"C:\Users\qxnq723\Desktop\Project 1\Coding\Bradford Data\UK_holiday.csv")
bank_hols.head()

,title,date,notes,bunting
0,New Year’s Day,2018-01-01,NaN,True
1,Good Friday,2018-03-30,NaN,False
2,Easter Monday,2018-04-02,NaN,True
3,Early May bank holiday,2018-05-07,NaN,True
4,Spring bank holiday,2018-05-28,NaN,True


In [139]:
#Check date range
bank_hols['date'].unique()

array(['2018-01-01', '2018-03-30', '2018-04-02', '2018-05-07',
       '2018-05-28', '2018-08-27', '2018-12-25', '2018-12-26',
       '2019-01-01', '2019-04-19', '2019-04-22', '2019-05-06',
       '2019-05-27', '2019-08-26', '2019-12-25', '2019-12-26',
       '2020-01-01', '2020-04-10', '2020-04-13', '2020-05-08',
       '2020-05-25', '2020-08-31', '2020-12-25', '2020-12-28',
       '2021-01-01', '2021-04-02', '2021-04-05', '2021-05-03',
       '2021-05-31', '2021-08-30', '2021-12-27', '2021-12-28',
       '2022-01-03', '2022-04-15', '2022-04-18', '2022-05-02',
       '2022-06-02', '2022-06-03', '2022-08-29', '2022-09-19',
       '2022-12-26', '2022-12-27', '2023-01-02', '2023-04-07',
       '2023-04-10', '2023-05-01', '2023-05-08', '2023-05-29',
       '2023-08-28', '2023-12-25', '2023-12-26', '2024-01-01',
       '2024-03-29', '2024-04-01', '2024-05-06', '2024-05-27',
       '2024-08-26', '2024-12-25', '2024-12-26', '2025-01-01',
       '2025-04-18', '2025-04-21', '2025-05-05', '2025-

The data includes all bank holidays between 2019 and 2025, thus we can create a variable in the footfall data to indicate whether or not a day falls on a bank holiday, using the one hot encoding technique (0 is no, 1 is yes).

In [140]:
#Convert datestamp to datetime
bank_hols['date'] = pd.to_datetime(bank_hols['date'])
#Create set of bank holiday dates
holiday_set = set(bank_hols['date'])

In [141]:
#Add column in footfall data (1= holiday, 0= normal day)
footfall['bank_hol'] = footfall['datestamp'].isin(holiday_set).astype(int)
#Check
footfall.head()

,Unnamed: 0,datestamp,estimated_actual_footfall,estimated_actual_footfall_rolling,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,Sin_week_of_year,Cos_week_of_year,Sin_month,Cos_month,bank_hol
0,2497,2019-01-01,530996.0,571980,2019,1,1,1,8.660254e-01,0.5,0.118273,0.992981,0.5,0.866025,1
1,2498,2019-01-02,568621.0,572734,2019,1,2,1,8.660254e-01,-0.5,0.118273,0.992981,0.5,0.866025,0
2,2499,2019-01-03,606939.0,538667,2019,1,3,1,1.224647e-16,-1.0,0.118273,0.992981,0.5,0.866025,0
3,2500,2019-01-04,508695.0,532787,2019,1,4,1,-8.660254e-01,-0.5,0.118273,0.992981,0.5,0.866025,0
4,2501,2019-01-05,468546.0,507700,2019,1,5,1,-8.660254e-01,0.5,0.118273,0.992981,0.5,0.866025,0


## Covid
The first covid lock down occured on the 23rd of March 2020 and the last covid restrictions were lifted on the 24 February 2022 in England. Thus, dates that fall within this period will be assigned a 1 in the 'covid' column.

In [142]:
#Create set of dates during covid times in UK
start= '2020-03-23'
end= '2022-02-24'

covid_dates = pd.date_range(start=start, end=end, freq='D')

#Add column in footfall data (1= covid, 0= normal day)
footfall['covid'] = footfall['datestamp'].isin(covid_dates).astype(int)
#Check
footfall.head()

,Unnamed: 0,datestamp,estimated_actual_footfall,estimated_actual_footfall_rolling,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,Sin_week_of_year,Cos_week_of_year,Sin_month,Cos_month,bank_hol,covid
0,2497,2019-01-01,530996.0,571980,2019,1,1,1,8.660254e-01,0.5,0.118273,0.992981,0.5,0.866025,1,0
1,2498,2019-01-02,568621.0,572734,2019,1,2,1,8.660254e-01,-0.5,0.118273,0.992981,0.5,0.866025,0,0
2,2499,2019-01-03,606939.0,538667,2019,1,3,1,1.224647e-16,-1.0,0.118273,0.992981,0.5,0.866025,0,0
3,2500,2019-01-04,508695.0,532787,2019,1,4,1,-8.660254e-01,-0.5,0.118273,0.992981,0.5,0.866025,0,0
4,2501,2019-01-05,468546.0,507700,2019,1,5,1,-8.660254e-01,0.5,0.118273,0.992981,0.5,0.866025,0,0


## Weather

This section collects daily weather data (average temperature, total precipitation, maximum wind speed) for Bradford over the study period (2019-2025) using the Open Meteo API.

In [143]:
pip install --update typing extensions

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\ProgramData\anaconda3\python.exe -m pip install [options] <requirement specifier> [package-index-options] ...
  c:\ProgramData\anaconda3\python.exe -m pip install [options] -r <requirements file> [package-index-options] ...
  c:\ProgramData\anaconda3\python.exe -m pip install [options] [-e] <vcs project url> ...
  c:\ProgramData\anaconda3\python.exe -m pip install [options] [-e] <local project path> ...
  c:\ProgramData\anaconda3\python.exe -m pip install [options] <archive url/path> ...

no such option: --update


In [144]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests

In [145]:
#Import packages
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [146]:
#Code provided by Open-Meteo API website

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 53.799999,
	"longitude": -1.750000,
	"start_date": "2019-01-01",
	"end_date": "2025-12-01",
	"daily": ["temperature_2m_mean", "precipitation_sum", "wind_speed_10m_max"],
}

responses = openmeteo.weather_api(url, params=params)

# Process location
response = responses[0]


# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(1).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(2).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max

daily_weather = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_weather)


Daily data
                           date  temperature_2m_mean  precipitation_sum  \
0    2019-01-01 00:00:00+00:00             6.470333           0.000000   
1    2019-01-02 00:00:00+00:00             2.616167           0.000000   
2    2019-01-03 00:00:00+00:00             0.699500           0.000000   
3    2019-01-04 00:00:00+00:00             2.376583           0.000000   
4    2019-01-05 00:00:00+00:00             4.080750           0.000000   
...                        ...                  ...                ...   
2522 2025-11-27 00:00:00+00:00            11.505750           1.700000   
2523 2025-11-28 00:00:00+00:00             7.959917           4.600000   
2524 2025-11-29 00:00:00+00:00             4.926583           9.000000   
2525 2025-11-30 00:00:00+00:00             3.503666           0.500000   
2526 2025-12-01 00:00:00+00:00             9.853666          16.900002   

      wind_speed_10m_max  
0              24.490587  
1              10.086427  
2               6

In [148]:
#Rename column
daily_weather= daily_weather.rename(columns= {'date': 'datestamp'})

#Convert datestamp (avoids having hour:min:secs) to timezone naive
daily_weather['datestamp'] = (pd.to_datetime(daily_weather['datestamp'])
                              .dt.tz_localize(None)
                              .dt.normalize())

In [149]:
daily_weather.head()

,datestamp,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,2019-01-01,6.470333,0.0,24.490587
1,2019-01-02,2.616167,0.0,10.086427
2,2019-01-03,0.699500,0.0,6.480000
3,2019-01-04,2.376583,0.0,15.629971
4,2019-01-05,4.080750,0.0,14.168641


In [150]:
#Merge the weather data and the footfall data based on dates
footfall_clean = footfall.merge(daily_weather[['datestamp', 'temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max']], on='datestamp')
footfall_clean.head()

,Unnamed: 0,datestamp,estimated_actual_footfall,estimated_actual_footfall_rolling,year,month,weekday,week_of_year,Sin_weekday,Cos_weekday,Sin_week_of_year,Cos_week_of_year,Sin_month,Cos_month,bank_hol,covid,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,2497,2019-01-01,530996.0,571980,2019,1,1,1,8.660254e-01,0.5,0.118273,0.992981,0.5,0.866025,1,0,6.470333,0.0,24.490587
1,2498,2019-01-02,568621.0,572734,2019,1,2,1,8.660254e-01,-0.5,0.118273,0.992981,0.5,0.866025,0,0,2.616167,0.0,10.086427
2,2499,2019-01-03,606939.0,538667,2019,1,3,1,1.224647e-16,-1.0,0.118273,0.992981,0.5,0.866025,0,0,0.699500,0.0,6.480000
3,2500,2019-01-04,508695.0,532787,2019,1,4,1,-8.660254e-01,-0.5,0.118273,0.992981,0.5,0.866025,0,0,2.376583,0.0,15.629971
4,2501,2019-01-05,468546.0,507700,2019,1,5,1,-8.660254e-01,0.5,0.118273,0.992981,0.5,0.866025,0,0,4.080750,0.0,14.168641


In [152]:
#Drop unneeded columns
footfall_clean = footfall_clean.drop(columns=['Unnamed: 0'])

Save the full cleaned dataset, ready for analysis:

In [153]:
footfall.to_csv('footfall_Met_clean')